# Almacenamiento e Integración de Datos (DW, Data Lake, Lakehouse, ETL/ELT)

Objetivos:
- Diferenciar Data Warehouse, Data Lake y Lakehouse y su rol en analítica/BI
- Comprender ETL vs ELT y cuándo usar cada enfoque
- Diseñar estrategias de integración para fuentes heterogéneas (estructuradas, semiestructuradas, no estructuradas)
- Practicar con ejemplos compactos de ingestión y transformación

## 1. Arquitecturas de Datos: DW vs Data Lake vs Lakehouse

| Atributo | Data Warehouse | Data Lake | Lakehouse |
|---|---|---|---|
| Esquema | Schema-on-Write | Schema-on-Read | Híbrido (medallion) |
| Tipos de datos | Estructurados | Todos (CSV, JSON, imágenes) | Todos |
| Casos | BI, reporting, KPIs | Ciencia de datos, ML, raw | BI + DS/ML unificados |
| Gobernanza | Alta, rígida | Variable | Alta con capas (Bronze/Silver/Gold) |
| Rendimiento | Alto en SQL analítico | Variable | Alto (columnar + cache) |

Notas clave:
- DW: Modelo dimensional, datos curados, rendimiento predecible.
- Lake: Flexibilidad, bajo costo, conserva raw.
- Lakehouse: Capas Medallion (Bronze→Silver→Gold), unifica almacenamiento y procesamiento analítico.

In [ ]:
# Diagrama simple (fuentes -> ingestión -> capas)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
fig, ax = plt.subplots(figsize=(12,6))
ax.axis('off')
# Fuentes
ax.text(0.1, 0.8, 'Fuentes', fontsize=12, fontweight='bold')
fuentes = ['DB OLTP','APIs','Files (CSV/JSON)','Streaming']
for i,s in enumerate(fuentes):
    ax.add_patch(mpatches.FancyBboxPatch((0.05, 0.65 - i*0.12), 0.25, 0.08, boxstyle='round', facecolor='#A0E7E5'))
    ax.text(0.175, 0.69 - i*0.12, s, ha='center', va='center')
# Ingestión
ax.add_patch(mpatches.FancyBboxPatch((0.38, 0.55), 0.25, 0.35, boxstyle='round', facecolor='#B4F8C8'))
ax.text(0.505, 0.72, 'Ingestión', ha='center', fontsize=12, fontweight='bold')
ax.text(0.505, 0.67, 'Batch/Streaming\nCDC, API, Files', ha='center')
# Lakehouse capas
ax.add_patch(mpatches.FancyBboxPatch((0.72, 0.62), 0.25, 0.08, boxstyle='round', facecolor='#FBE7C6'))
ax.text(0.845, 0.66, 'Bronze (Raw)', ha='center')
ax.add_patch(mpatches.FancyBboxPatch((0.72, 0.49), 0.25, 0.08, boxstyle='round', facecolor='#FBE7C6'))
ax.text(0.845, 0.53, 'Silver (Curado)', ha='center')
ax.add_patch(mpatches.FancyBboxPatch((0.72, 0.36), 0.25, 0.08, boxstyle='round', facecolor='#FBE7C6'))
ax.text(0.845, 0.40, 'Gold (BI/ML)', ha='center')
# Flechas
for i in range(len(fuentes)):
    ax.annotate('', xy=(0.38, 0.59), xytext=(0.3, 0.69 - i*0.12), arrowprops=dict(arrowstyle='->'))
ax.annotate('', xy=(0.72, 0.66), xytext=(0.63, 0.66), arrowprops=dict(arrowstyle='->'))
ax.annotate('', xy=(0.72, 0.53), xytext=(0.63, 0.66), arrowprops=dict(arrowstyle='->'))
ax.annotate('', xy=(0.72, 0.40), xytext=(0.63, 0.66), arrowprops=dict(arrowstyle='->'))
plt.title('Arquitectura Lakehouse con Capas Medallion', pad=20)
plt.show()

## 2. ETL vs ELT

- ETL: Extraer → Transformar → Cargar (transformación fuera del destino).
  - Útil cuando el destino (DW) es costoso/limitado para transformar.
- ELT: Extraer → Cargar → Transformar (transformación en el destino).
  - Útil en plataformas MPP/lakehouse modernas (BigQuery, Snowflake, Databricks).

Criterios:
- Volumen/velocidad, costo, gobernanza, herramientas, latencia aceptable, habilidades del equipo.

In [ ]:
# Mini demo: ETL (transformo con pandas antes de cargar)
import pandas as pd
clientes_csv = pd.DataFrame({
    'id': [1,2,3],
    'nombre': ['Ana','Luis','María'],
    'email': ['ana@example.com','luis@example.com','maria@example.com']
})

ventas_json = [
    {'idVenta': 10, 'clienteId': 1, 'items': [{'sku':'P1','q':1,'price':900}]},
    {'idVenta': 11, 'clienteId': 2, 'items': [{'sku':'P2','q':2,'price':25}]},
]
raw_ventas = pd.json_normalize(ventas_json, record_path=['items'], meta=['idVenta','clienteId'])
raw_ventas.rename(columns={'q':'cantidad','price':'precio'}, inplace=True)
# Transformaciones (limpieza, derivaciones)
raw_ventas['importe'] = raw_ventas['cantidad'] * raw_ventas['precio']
tabla_hechos = raw_ventas.merge(clientes_csv, left_on='clienteId', right_on='id', how='left')[['idVenta','sku','cantidad','importe','clienteId','nombre']]
print('Hechos (ETL):\n', tabla_hechos)
print('
Carga: se escribiría en DW/Lake (omitido para mantener dependencias mínimas)')

# Mini demo: ELT (cargo raw y transformo después)
raw_clientes = clientes_csv.copy()
raw_ventas_bronze = pd.DataFrame(ventas_json)
# 'Transformación' posterior (simulada)
items_exp = pd.json_normalize(ventas_json, record_path=['items'], meta=['idVenta','clienteId'])
items_exp['importe'] = items_exp['q'] * items_exp['price']
elt_hechos = items_exp.merge(raw_clientes, left_on='clienteId', right_on='id', how='left')[['idVenta','sku','q','importe','clienteId','nombre']]
elt_hechos.rename(columns={'q':'cantidad'}, inplace=True)
print('
Hechos (ELT):\n', elt_hechos)
print('
Diferencia conceptual: en ELT primero persisto raw (bronze) y luego derivo silver/gold.')

## 3. Estrategias de Integración de Múltiples Fuentes

Patrones de ingestión:
- Batch (ventanas programadas)
- Streaming (near real-time)
- Micro-batch (compromiso)

Técnicas clave:
- CDC (Change Data Capture): log-based, timestamp, triggers
- API Pull/Push, Webhooks
- Archivos (S3/ADLS/GCS), colas/mensajería (Kafka, Pub/Sub)
- Idempotencia y reintentos, compaction, watermarking
- Validación y calidad: esquemas, reglas, perfiles
- Catálogo de datos y lineage
- Enriquecimiento y master data (MDM)

Formatos comunes:
- CSV (simple), JSON (flexible), Avro/Parquet/ORC (columnar, eficiencia), binarios (imágenes/audio)

Esquemas y evolución:
- Gestión de cambios: schema registry, versionado, compatibilidad (backward/forward)

In [ ]:
# Normalizar JSON anidado (fuente semiestructurada)
import pandas as pd
pedido_raw = {
    'pedidoId': 1001,
    'cliente': {'id': 7, 'nombre': 'Lucía', 'pais': 'AR'},
    'items': [
        {'sku': 'B-1', 'cantidad': 1, 'precio': 120.0},
        {'sku': 'B-2', 'cantidad': 2, 'precio': 15.0}
    ]
}
items = pd.json_normalize(pedido_raw, record_path=['items'], meta=[['cliente','id'], ['cliente','nombre'], 'pedidoId'])
items.rename(columns={
: 
, 
: 
}, inplace=True)
items['importe'] = items['cantidad'] * items['precio']
print(items)
print('
Estrategia: guardar raw (JSON) en Bronze y tabla aplanada en Silver.')

## 4. Diseño de Flujos: Orquestación y Calidad

- Orquestación: Airflow, Prefect, Dagster (dependencias, scheduling, retries).
- Calidad: expectativas (Great Expectations), reglas SQL, perfiles estadísticos.
- Observabilidad: logs, métricas, SLA/SLO, datos de control (row counts).
- Seguridad: control de acceso, masking, tokenización, cifrado en tránsito/en reposo.
- Costos: almacenamiento frío vs caliente; compresión, particionado, TTL/retención.

## 5. Ejercicio Práctico (Guided)

Escenario: Retail omnicanal con fuentes POS (OLTP), e-commerce (API), CRM (DB), y clicstream (stream).

Tareas:
1) Define arquitectura objetivo (DW, Lake, Lakehouse) y justifica.
2) Propón pipeline de ingestión (batch/stream, CDC, formatos).
3) Diseña capas Bronze/Silver/Gold con ejemplos de tablas.
4) Elige ETL o ELT para cada flujo y explica razones.
5) Lista 5 checks de calidad (por capa) y 3 métricas de observabilidad.

In [ ]:
# Plantilla de respuesta
plantilla = '''
# Solución (borrador)

1) Arquitectura: [Describe y justifica]
2) Ingestión: [Batch/Streaming, CDC, formatos, colas]
3) Capas: [Bronze tablas raw, Silver aplanadas/curadas, Gold data marts]
4) ETL/ELT por flujo: [Razones: costo, latencia, skills, plataforma]
5) Calidad/Observabilidad: [Checks: esquema, nulos, duplicados, rangos, conteos; Métricas: latencia, throughput, fallos]
'''
print(plantilla)

## 6. Resumen

- DW, Lake y Lakehouse responden a necesidades distintas; Lakehouse integra lo mejor de ambos mundos.
- ETL vs ELT es una decisión de arquitectura y plataforma.
- Integración efectiva requiere patrones sólidos (CDC, streaming, catálogo, calidad).
- Capas Medallion ayudan a gobernanza, repetibilidad y performance.

Próximo: patrones Lambda/Kappa y orquestación avanzada.